# M2-B2 — Biais du modèle d'annulation (binôme)

Le calcul est fourni ; les constats, les choix et leur interprétation sont à rédiger dans les cellules ✏️.

In [5]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
from features import INPUT_FEATURES
from modele import entrainer, mesures_par_groupe

train = pd.read_csv(ROOT / "data/reservations_train.csv")
test = pd.read_csv(ROOT / "data/reservations_test.csv")
v21 = joblib.load(ROOT / "model/cancellation_model_v2.1.0.joblib")
len(train), len(test)

(58199, 19694)

## Tâche 2 — Mesurer

Groupes : clients portugais (`PRT`) et autres pays (pays manquant compris).

In [6]:
groupe = test["country"].eq("PRT").map({True: "Portugal", False: "Autres pays"})
mesures_par_groupe(v21, test, INPUT_FEATURES, groupe)

,effectif,annulations_reelles,taux_alertes,rappel,fausses_alertes,precision
Autres pays,12871.0,0.258,0.223,0.530,0.117,0.612
Portugal,6823.0,0.620,0.744,0.931,0.438,0.776


### ✏️ Constats

Que dit chaque colonne, en une phrase compréhensible par le service réservations ? Parmi les clients qui **viendront**, qui reçoit le plus souvent une alerte à tort ?

- *annulations_reelles* : les clients portugais annulent 62 % de leurs réservations, contre 26 % pour les autres pays.
- *taux_alertes* : le modèle signale 74 % des réservations portugaises, contre 22 % des autres. 
- *rappel* : le modèle repère 93 % des annulations portugaises, mais seulement 53 % des annulations des autres pays.
- *fausses_alertes* : parmi les clients qui viennent, 44 % des Portugais sont signalés à tort, contre 12 % des autres.
- *precision* : sur 100 réservations portugaises signalées, 77 sont réellement annulées. Pour les autres pays, 61 sur 100.

## Tâche 3 — Débattre

| | Arguments |
|---|---|
| Position A : l'écart est acceptable | Les taux réels diffèrent : 62 % d'annulations pour le Portugal, 26 % pour les autres pays.<br>Une alerte sur un client portugais est plus souvent juste (précision 78 % contre 61 %).<br>Le modèle repère 93 % des annulations portugaises : le service en rate peu. |
| Position B : l'écart est injuste | Parmi les clients qui viennent, un Portugais est signalé à tort plus de 3 fois plus souvent (44 % contre 12 %).<br>Le modèle accentue l'écart réel : 74 % d'alertes pour 62 % d'annulations côté Portugal, 22 % pour 26 % côté autres pays.<br>Le pays n'est pas une cause d'annulation et il est proche de la nationalité. |
| Décision du binôme, si l'alerte déclenche une relance | Oui|
| Décision du binôme, si l'alerte déclenche un acompte | Acceptable uniquement sous forme d'acompte remboursable, demandé lors de la relance.<br>Le client qui vient récupère son acompte : le préjudice est limité. |

## Tâche 4 — Atténuer

`entrainer(train, colonnes)` réentraîne la même recette que la v2.1 (seuil 0,2) sur les colonnes données.

In [7]:
# À compléter : réentraîner sans `country`, mesurer par groupe, comparer la performance globale à celle de la v2.1

### À investiguer

Peut-on deviner le pays sans la colonne `country` ? Choisir une méthode pour répondre, et dire quelles variables sont en cause.

In [8]:
# À compléter

### ✏️ Interprétation

1. L'écart a-t-il disparu, diminué, changé de sens ? Pour quelles mesures ?
2. Qui gagne et qui perd au retrait de `country` ?
3. Quel est le coût en performance, et est-il acceptable ?
4. Faut-il interroger le client sur une autre variable ? Laquelle ?

*…*